## Imports

In [ ]:
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import visualkeras

from keras.callbacks import EarlyStopping

from keras.utils import to_categorical

In [ ]:
print("Available Physical Devices: ", tf.config.list_physical_devices())

## Functions

In [ ]:
def show_image(image):

    plt.imshow(image)
    plt.axis("off")
    plt.show()

## Preprocessing

#### LOADING DATA

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar100.load_data(label_mode="coarse")
assert X_train.shape == (50000, 32, 32, 3)
assert X_test.shape == (10000, 32, 32, 3)
assert y_train.shape == (50000, 1)
assert y_test.shape == (10000, 1)

In [ ]:
show_image(X_train[4])

In [ ]:
y_train = y_train.flatten()
y_test = y_test.flatten()

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

In [ ]:
set(y_train)

In [ ]:
y_train = to_categorical(y_train, num_classes=20)
y_test = to_categorical(y_test, num_classes=20)

In [ ]:
len(X_train), len(y_train), len(X_test), len(y_test)

## Models

### Model 1

#### Creation

In [ ]:
data_augmentation_layer = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.2),
    ]
)

In [ ]:
inputs = keras.Input(shape=(32, 32, 3, ))

x = data_augmentation_layer(inputs)

x = keras.layers.Rescaling(1./255)(x)
x = keras.layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = keras.layers.MaxPooling2D(pool_size=2)(x)
x = keras.layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = keras.layers.Flatten()(x)
x = keras.layers.Dropout(0.4)(x)

outputs = keras.layers.Dense(20, activation="softmax")(x)
model_1 = keras.Model(inputs = inputs, outputs = outputs)
model_1.summary()

In [ ]:
visualkeras.layered_view(model_1, legend=True)

#### Compilation

In [ ]:
model_1.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics = ["accuracy"]
)

#### Tranining

In [ ]:
early_stopping_cb = EarlyStopping(patience=15, monitor="val_loss", mode="min", restore_best_weights=True)

In [ ]:
history_model_1 = model_1.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=200,
    callbacks=[early_stopping_cb]
)

In [ ]:
model_1.save("Model1.h5")

In [ ]:
pd.DataFrame(history_model_1.history).plot()

#### Testing

In [ ]:
model_1.evaluate(X_test, y_test)

#### Results